# 기계학습 응용 실습 · 분류와 군집 — 코어가 없는 시추공의 암상을 맞힐 수 있을까 (10주차 · 2026. 11. 23.)

기초지질데이터과학 · **오늘은 제출 없음.** 5번(Part C, 내 포트폴리오 자료에 모델 하나)은 여러분의 포트폴리오 작성에 활용하세요. 노트북으로 이어 씁니다 — 이 노트북을 내 드라이브에 저장해 두세요(파일 → 드라이브에 사본 저장).

**이름 · 학번:**　　　　　　　　**쓴 AI:**

### 데이터 명세서 (교육용 합성 자료 — 실제 검층 자료의 물리 관계와 흔한 문제를 그대로 넣었습니다)

| 항목 | 내용 |
|---|---|
| ① 무엇인가 | 한 퇴적분지에서 판 시추공 4개(A–D)의 물성을 깊이 **0.5 m 간격**으로 잰 값. **A · B · C공**은 코어를 회수해 지질학자가 암상을 기재했다 → `core_wells_ABC.csv`(600행, `암상` 열 있음). **D공**은 최근에 판 공으로 코어 없이 검층만 했다 → `core_well_D.csv`(200행, 암상 모름). |
| ② 열 · 단위 | `시추공` · `깊이_m` · `밀도_gcc`(g/cm³) · `P파속도_mps`(m/s) · `공극률_pct`(%) · `감마선_API`(자연 감마선 — 점토의 K · Th · U, **셰일이 높다**) · `대자율_mSI`(10⁻³ SI — 자철석이 많을수록 크다, **현무암이 수십~수천 배**) · `암상`(A–C공만: 사암 · 셰일 · 석회암 · 현무암) |
| ③ 알려진 함정 | 검층의 수직 분해능은 약 1.5 m — **층 경계에서는 위아래 층의 값이 섞인다** · 0.5 m 간격이라 **같은 층의 이웃 시료는 서로 아주 닮았다** · **D공 검층은 다른 업체가 했다** — 열 이름은 맞췄지만 단위와 보정이 같은지는 확인하지 않았다 |
| ④ 답할 수 있는 / 없는 질문 | 있음: 물성으로 암상을 얼마나 맞히나, D공의 암상 기둥(예측), 물성이 비슷한 층의 무리 · 없음: 퇴적 환경과 지층의 나이(물성에 없음), 시추공 사이 지층의 연결(공 4개로는 부족), 석유가 있는가 |
| ⑤ AI가 틀리면 무엇을 보고 아는가 | D공 예측이 **한 암상으로 몰린다** · 정확도가 **99–100 %**처럼 너무 좋다 · 예측과 물성이 **모순**된다(대자율이 낮은 현무암, 감마선이 낮은 셰일) · 군집 중심값이 아래 표의 범위와 맞지 않는다 |

**이 분지에서 암상별로 흔한 값** (A–C공의 5–95 % 범위 — 군집을 해석할 때 씁니다)

| 암상 | 밀도 (g/cm³) | P파 속도 (m/s) | 공극률 (%) | 감마선 (API) | 대자율 (10⁻³ SI) |
|---|---|---|---|---|---|
| 사암 | 2.1–2.3 | 2,900–3,600 | 20–34 | 25–85 | 0.03–0.3 |
| 셰일 | 2.2–2.5 | 2,900–3,400 | 15–28 | 65–120 | 0.1–0.4 |
| 석회암 | 2.5–2.7 | 4,600–5,800 | 3–12 | 5–40 | 0.005–0.04 |
| 현무암 | 2.7–2.9 | 4,600–5,500 | 4–11 | 5–40 | 7–50 |

**순서**: ① 원자료 점검(1번) → ② **Part A** 분류: 프롬프트를 그대로 AI에게(CSV 두 개 첨부) → 답 기록 → 3번 칸으로 검증 → ③ **Part B** 군집: 프롬프트 → 4번 칸으로 검증 → ④ **Part C** 내 자료에 모델 하나(5번) → 11/30 중간 2.
★ = 꼭 · ☆ = 시간이 남으면. 시작하기 전에 왼쪽 폴더 아이콘에서 CSV 두 개를 올리세요.


## 0. 도구 (실행만 하세요)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")
from IPython.display import display
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split, cross_val_score, cross_val_predict, LeaveOneGroupOut, StratifiedKFold, KFold
from sklearn.metrics import accuracy_score, confusion_matrix, silhouette_score
from sklearn.cluster import KMeans

ABC_FILE, D_FILE = "core_wells_ABC.csv", "core_well_D.csv"
FEATURES = ["밀도_gcc", "P파속도_mps", "공극률_pct", "감마선_API", "대자율_mSI"]
LITHS = ["사암", "셰일", "석회암", "현무암"]
LITH_EN = {"사암": "sandstone", "셰일": "shale", "석회암": "limestone", "현무암": "basalt"}
COLORS = {"사암": "#E3B448", "셰일": "#5B6770", "석회암": "#6FA8DC", "현무암": "#A33A2E"}
RANGES = {"밀도_gcc": (1.9, 3.1), "P파속도_mps": (1500, 7000), "공극률_pct": (0, 45),
          "감마선_API": (0, 250), "대자율_mSI": (0, 100)}


def load():
    """원자료 두 개를 손대지 않고 읽는다."""
    return pd.read_csv(ABC_FILE), pd.read_csv(D_FILE)


def check_ranges(df, name=""):
    """열마다 최솟값 · 중앙값 · 최댓값을 데이터 명세서의 정상 범위와 나란히 놓는다."""
    rows = []
    for c, (lo, hi) in RANGES.items():
        if c in df:
            v = df[c]
            rows.append((c, v.min(), v.median(), v.max(), f"{lo}–{hi}", int(((v < lo) | (v > hi)).sum())))
    t = pd.DataFrame(rows, columns=["열", "최솟값", "중앙값", "최댓값", "정상 범위", "범위 밖 행"])
    if name:
        print(name)
    display(t)


def _model(k, scale):
    return make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=k)) if scale else KNeighborsClassifier(n_neighbors=k)


def knn_report(df, k=5, scale=True, split="random", features=FEATURES):
    """k-NN 정확도. split='random': 무작위 70/30(층화) · split='well': 시추공 하나를 빼고 나머지로 학습해 뺀 공을 예측(공마다 반복).
    split='well'이면 예측값 배열을 돌려준다(혼동행렬용)."""
    X, y = df[features], df["암상"]
    tag = f"k={k} · 표준화 {'O' if scale else 'X'}"
    if split == "random":
        Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
        m = _model(k, scale).fit(Xtr, ytr)
        tr, te = accuracy_score(ytr, m.predict(Xtr)), accuracy_score(yte, m.predict(Xte))
        print(f"[무작위 70/30 · {tag}]  훈련 정확도 {tr:.3f} · 검증 정확도 {te:.3f}")
        return None
    w = df["시추공"]
    pred = cross_val_predict(_model(k, scale), X, y, cv=LeaveOneGroupOut(), groups=w)
    each = " · ".join(f"{g}공 {accuracy_score(y[w == g], pred[w == g]):.3f}" for g in sorted(w.unique()))
    print(f"[시추공 단위 · {tag}]  전체 {accuracy_score(y, pred):.3f}  ({each})")
    return pred


def k_curve(df, ks=range(1, 32, 2), features=FEATURES):
    """k에 따른 검증 정확도 — 무작위 5겹 교차검증 vs 시추공 단위(표준화함)."""
    X, y, w = df[features], df["암상"], df["시추공"]
    r = [cross_val_score(_model(k, True), X, y, cv=StratifiedKFold(5, shuffle=True, random_state=0)).mean() for k in ks]
    g = [cross_val_score(_model(k, True), X, y, cv=LeaveOneGroupOut(), groups=w).mean() for k in ks]
    tab = pd.DataFrame({"k": list(ks), "무작위 5겹": np.round(r, 3), "시추공 단위": np.round(g, 3)}).set_index("k")
    ax = tab.rename(columns={"무작위 5겹": "random 5-fold", "시추공 단위": "leave-one-well-out"}).plot(marker="o", figsize=(6, 3.2))
    ax.set_xlabel("k (number of neighbours)"); ax.set_ylabel("validation accuracy"); ax.grid(alpha=.3); plt.show()
    return tab


def confusion(df, pred, labels=LITHS):
    """혼동행렬(행 = 실제 암상, 열 = 예측)과 암상별 재현율 · 정확도 · 최다 범주 기준선."""
    y = df["암상"]
    cm = pd.DataFrame(confusion_matrix(y, pred, labels=labels), index=[f"실제 {l}" for l in labels], columns=[f"예측 {l}" for l in labels])
    display(cm)
    rec = {l: round(float(cm.iloc[i, i] / cm.iloc[i].sum()), 3) for i, l in enumerate(labels) if cm.iloc[i].sum()}
    print("암상별 재현율(실제 그 암상 중 맞힌 비율):", rec)
    print(f"정확도 {accuracy_score(y, pred):.3f} · 기준선(늘 '{y.value_counts().idxmax()}'이라고 답하면) {y.value_counts(normalize=True).max():.3f}")


def predict_D(train_df, d_df, k=5, scale=True, features=FEATURES):
    """A–C공 전체로 학습해 D공 암상을 예측한다. 예측값 배열을 돌려준다."""
    m = _model(k, scale).fit(train_df[features], train_df["암상"])
    p = m.predict(d_df[features])
    print("D공 예측 암상 개수:", pd.Series(p).value_counts().to_dict())
    return p


def depth_strip(depth, columns, title=""):
    """깊이에 따른 암상(또는 군집) 기둥을 나란히. columns = {"이름": 값 배열}."""
    fig, axes = plt.subplots(1, len(columns), figsize=(1.6 * len(columns) + 1, 6), sharey=True)
    axes = np.atleast_1d(axes)
    dz = np.median(np.diff(np.sort(np.asarray(depth))))
    for ax, (name, vals) in zip(axes, columns.items()):
        vals = np.asarray(vals)
        keys = list(dict.fromkeys(vals))
        pal = plt.cm.tab10.colors
        for i, kk in enumerate(keys):
            sel = vals == kk
            col = COLORS.get(kk, pal[i % 10])
            ax.barh(np.asarray(depth)[sel], 1, height=dz, color=col, label=LITH_EN.get(kk, str(kk)))
        ax.set_xticks([]); ax.set_title(name, fontsize=9)
        ax.legend(fontsize=7, loc="lower center", bbox_to_anchor=(0.5, -0.25), ncol=1, frameon=False)
    axes[0].invert_yaxis(); axes[0].set_ylabel("depth (m)")
    if title:
        fig.suptitle(title, fontsize=10)
    plt.tight_layout(); plt.show()


def _prep(X, scale=True, log_ms=False):
    Z = X.copy()
    if log_ms and "대자율_mSI" in Z:
        Z["대자율_mSI"] = np.log10(Z["대자율_mSI"].clip(lower=1e-4))
    return StandardScaler().fit_transform(Z) if scale else Z.values


def kmeans_scan(X, ks=range(2, 8), scale=True, log_ms=False):
    """k마다 군집 내 제곱합(엘보)과 실루엣 점수."""
    Z = _prep(X, scale, log_ms); rows = []
    for k in ks:
        km = KMeans(n_clusters=k, n_init=10, random_state=0).fit(Z)
        rows.append((k, km.inertia_, silhouette_score(Z, km.labels_)))
    tab = pd.DataFrame(rows, columns=["k", "군집 내 제곱합", "실루엣"]).set_index("k").round(3)
    fig, ax = plt.subplots(1, 2, figsize=(8, 2.8))
    ax[0].plot(tab.index, tab["군집 내 제곱합"], "o-"); ax[0].set_xlabel("k"); ax[0].set_ylabel("within-cluster SS (elbow)")
    ax[1].plot(tab.index, tab["실루엣"], "o-", color="C1"); ax[1].set_xlabel("k"); ax[1].set_ylabel("silhouette")
    for a in ax: a.grid(alpha=.3)
    plt.tight_layout(); plt.show()
    return tab


def kmeans_centers(df, k, features=FEATURES, scale=True, log_ms=False):
    """k-means 군집 번호와, 군집마다 원래 단위의 중앙값(해석용)."""
    Z = _prep(df[features], scale, log_ms)
    lab = KMeans(n_clusters=k, n_init=10, random_state=0).fit_predict(Z)
    cen = df[features].assign(군집=lab).groupby("군집").median().round(2)
    cen.insert(0, "시료 수", pd.Series(lab).value_counts().sort_index().values)
    display(cen)
    return lab


# ---- Part C: 내 자료용 ----
def baseline(y):
    """기준선 — 범주면 '늘 가장 많은 범주로 답하기'의 정확도, 수치면 '늘 평균으로 답하기'의 RMSE."""
    y = pd.Series(y).dropna()
    if y.dtype.kind in "biuf" and y.nunique() > 10:
        print(f"[수치 목표] 늘 평균({y.mean():.3g})으로 답할 때 RMSE = {y.std(ddof=0):.3g}")
    else:
        print(f"[범주 목표] 늘 '{y.value_counts().idxmax()}'(으)로 답할 때 정확도 = {y.value_counts(normalize=True).max():.3f}")


def quick_knn(df, target, features, k=5, groups=None):
    """내 자료로 k-NN 분류(표준화 포함) — 5겹 교차검증 정확도. groups에 열 이름을 주면 그 단위로 떼어 검증(예: 관측소 · 시추공 · 화산)."""
    sub = df[features + [target] + ([groups] if groups else [])].dropna()
    X, y = sub[features], sub[target]
    cv = LeaveOneGroupOut() if groups else StratifiedKFold(5, shuffle=True, random_state=0)
    pred = cross_val_predict(_model(k, True), X, y, cv=cv, groups=sub[groups] if groups else None)
    print(f"행 {len(sub)} · 특성 {len(features)}개 · k={k} · 검증: {'그룹 단위(' + groups + ')' if groups else '무작위 5겹'}")
    baseline(y)
    labels = list(y.value_counts().index)
    display(pd.DataFrame(confusion_matrix(y, pred, labels=labels), index=labels, columns=labels))
    print(f"검증 정확도 {accuracy_score(y, pred):.3f}")
    return pred


print("도구 준비 완료")


## 1. 원자료 점검 (5분)

In [ ]:
abc, d = load()
print("A–C공:", abc.shape, "· D공:", d.shape)
print(abc.groupby(["시추공", "암상"]).size().unstack(fill_value=0))
fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
for l in LITHS:
    s = abc[abc["암상"] == l]
    ax[0].scatter(s["감마선_API"], s["밀도_gcc"], s=9, color=COLORS[l], label=LITH_EN[l])
    ax[1].scatter(s["P파속도_mps"], s["대자율_mSI"], s=9, color=COLORS[l], label=LITH_EN[l])
ax[0].set_xlabel("gamma ray (API)"); ax[0].set_ylabel("density (g/cc)"); ax[0].legend(fontsize=8)
ax[1].set_xlabel("P-wave velocity (m/s)"); ax[1].set_ylabel("magnetic susceptibility (1e-3 SI)"); ax[1].set_yscale("log")
for a in ax: a.grid(alpha=.3)
plt.tight_layout(); plt.show()


## 2. Part A — k-NN 분류 · AI에게 묻기 (5분)

아래 프롬프트를 **그대로** 보내고 CSV 두 개를 첨부하세요(첨부가 안 되면 처음 몇십 행을 붙여 넣기). 후속 질문은 자유지만, **처음 답은 꼭 기록**합니다.

> 첨부한 시추 코어 물성 자료(core_wells_ABC.csv)로 암상(사암 · 셰일 · 석회암 · 현무암)을 분류하는 k-NN 모델을 파이썬(scikit-learn)으로 만들고 정확도를 구해 줘. 그다음 새 시추공 자료(core_well_D.csv)의 암상을 예측해 줘.

**AI의 답 기록** (아래를 채우세요)
- 입력(특성)으로 쓴 열:
- 표준화를 했나? 했다면 무엇에 맞춰(fit) 했나 — 훈련 자료 · 전체 · D공 따로:
- 훈련 / 검증을 어떻게 나눴나(비율, 무작위?):
- k는 얼마이고 어떻게 정했나:
- AI가 보고한 정확도:
- D공 예측 결과(암상별 개수):


In [ ]:
# AI가 준 코드를 여기에 붙여 실행하세요. 파일 이름은 그대로(core_wells_ABC.csv, core_well_D.csv).


## 3. 검증 — AI의 분류를 확인한다 (25분)

### 3-1 ★ D공 자료부터 — 단위가 같은가
명세서 ③: D공은 다른 업체 자료. 두 파일의 값을 정상 범위와 나란히 봅니다.

In [ ]:
check_ranges(abc, "A–C공")
check_ranges(d, "D공")


D공의 어느 열이 범위를 벗어났나? A–C공과 **몇 배** 차이인가? 아래 칸에 **한 줄**을 써서 고친 표 `d_fix`를 만드세요. AI의 D공 예측이 이상했다면 이것 때문인지 확인합니다.

In [ ]:
d_fix = d.copy()
# ↓ 여기에 한 줄 — D공의 잘못된 단위를 g/cm³로 바꾸기

check_ranges(d_fix, "D공(고친 뒤)")


### 3-2 ★ 표준화 — 거리는 단위를 탄다
k-NN은 **거리**로 이웃을 고릅니다. 열마다 퍼짐(표준편차)이 수천 배 다르면 어느 열이 거리를 좌우할까요?

In [ ]:
print("열마다 표준편차:", abc[FEATURES].std().round(3).to_dict())
knn_report(abc, k=5, scale=False, split="random")
knn_report(abc, k=5, scale=True, split="random")


### 3-3 ★ 나누는 법 — 같은 층의 이웃이 시험지에 들어가면
무작위 70/30으로 나누면 같은 층의 0.5 m 이웃이 훈련과 검증에 갈라져 들어갑니다 — 시험 문제를 미리 본 셈입니다. 우리가 실제로 하려는 일은 **처음 보는 시추공(D공)**의 예측이므로, **시추공 하나를 통째로 빼고** 나머지 둘로 배워 뺀 공을 맞히는 검증이 그 상황과 같습니다(9주차 다섯 질문의 3번 '나누는 법', 11/16의 자기상관).

In [ ]:
knn_report(abc, k=5, scale=True, split="random")
pred_well = knn_report(abc, k=5, scale=True, split="well")


### 3-4 ☆ k 고르기 — 무엇으로 고르나
k마다 검증 정확도를 두 방식으로 봅니다. k를 바꿔서 생기는 차이와, 나누는 법을 바꿔서 생기는 차이 중 어느 쪽이 큰가요?

In [ ]:
k_curve(abc)


### 3-5 ★ 정확도만 보지 말 것 — 혼동행렬과 기준선
정확도 하나로는 **무엇을 무엇으로 틀렸는지** 모릅니다. 시추공 단위 검증(3-3)의 예측으로 혼동행렬을 봅니다.

In [ ]:
confusion(abc, pred_well)


### 3-6 ★ D공 예측 — 깊이 기둥으로 보기
A–C공 전체로 배워 **고친 D공**(`d_fix`)의 암상을 예측합니다. 대자율이 큰 구간이 현무암으로, 감마선이 높은 구간이 셰일로 나왔나요? 물성과 예측이 모순되는 곳을 찾아보세요.

In [ ]:
pred_D = predict_D(abc, d_fix, k=5)
depth_strip(d_fix["깊이_m"], {"k-NN": pred_D}, "D well — predicted lithology")


### 해석 (Part A)
- **Q1.** AI의 D공 예측(또는 고치기 전 `predict_D(abc, d)`)은 어떻게 나왔고, 무엇을 보고 이상하다는 것을 알았나? 원인과 고친 방법:
- **Q2.** D공에 대해 기대할 정확도로 **무작위 분할**과 **시추공 단위** 중 어느 숫자를 보고하겠나? 숫자와 이유:
- **Q3.** 가장 많이 헷갈린 두 암상은? 어떤 물성이 겹쳐서인가(명세서 표의 근거):


## 4. Part B — k-means 군집 · 라벨 없이 묶으면 (20분)

D공에는 코어가 없습니다. 라벨 없이 **물성이 비슷한 시료끼리** 묶으면 어떤 무리가 나올까요?

> 새 시추공 자료(core_well_D.csv)를 k-means로 군집 분석하는 파이썬 코드를 짜 줘. 군집 수는 엘보 방법으로 정하고, 각 군집이 어떤 암석인지 해석해 줘. (밀도 열이 kg/m³로 잘못 들어가 있어서 g/cm³로 고친 표를 쓸 거야.)

**AI의 답 기록**
- 표준화를 했나:
- 엘보로 고른 k:
- 각 군집의 해석(무엇을 근거로):


In [ ]:
# AI 코드를 붙이고, 파일을 읽는 줄은 df = d_fix.copy() 로 바꿔 실행하세요(밀도를 고친 D공).


### 4-1 ★ k 고르기 — 엘보와 실루엣
군집 내 제곱합(엘보)은 k가 커질수록 늘 줄어듭니다. 실루엣은 '내 군집과 얼마나 가깝고 옆 군집과 얼마나 먼가'(−1 ~ 1, 클수록 잘 갈림). 두 기준이 같은 k를 가리키나요?

In [ ]:
scan = kmeans_scan(d_fix[FEATURES], scale=True)
scan


### 4-2 ★ 군집을 암석으로 읽기 — 중심값을 명세서 표와 대조
군집 번호(0, 1, 2 …)에는 뜻이 없습니다. 군집마다 **원래 단위의 중앙값**을 데이터 명세서의 '암상별로 흔한 값' 표와 비교해 이름을 붙이세요. k = 3과 k = 4는 무엇이 다른가요?

In [ ]:
print("k = 3"); lab3 = kmeans_centers(d_fix, k=3)
print("k = 4"); lab4 = kmeans_centers(d_fix, k=4)


### 4-3 ★ 두 방법 맞대기 — k-NN(라벨로 배움) vs k-means(라벨 없이 묶음)
서로 다른 방법이 같은 답을 내면 믿을 만하고, 엇갈리는 곳은 다시 볼 곳입니다.

In [ ]:
display(pd.crosstab(pd.Series(lab4, name="k-means 군집(k=4)"), pd.Series(pred_D, name="k-NN 예측")))
depth_strip(d_fix["깊이_m"], {"k-NN": pred_D, "k-means k=4": lab4}, "D well — two methods")


### 4-4 ☆ 표준화하지 않으면 · 대자율을 로그로 보면
대자율은 암석마다 수십~수천 배 차이 납니다(명세서 표). 이런 열은 흔히 로그로 바꿔 씁니다.

In [ ]:
print("표준화 X"); kmeans_scan(d_fix[FEATURES], scale=False)
print("표준화 O + 대자율 log10"); kmeans_scan(d_fix[FEATURES], scale=True, log_ms=True)


### 4-5 ☆ 엇갈린 시료의 감마선 — D공 검층기는 A–C공과 같게 읽을까?
k-NN은 '셰일', k-means는 '사암 쪽 무리'라고 한 시료들을 모아 봅니다. 감마선이 거의 없는 석회암 · 현무암에서도 두 자료를 비교해 보세요.

In [ ]:
cen = d_fix[FEATURES].assign(c=lab4).groupby("c").median()
clastic = cen[cen["P파속도_mps"] < 4000]
sand_c = clastic["감마선_API"].idxmin()                       # 쇄설암 중 감마선이 낮은 쪽 = 사암 쪽 무리
dis = (pd.Series(pred_D) == "셰일").values & (lab4 == sand_c)
print(f"엇갈린 시료 {dis.sum()}개 · 감마선 {d_fix.loc[dis, '감마선_API'].min():.0f}–{d_fix.loc[dis, '감마선_API'].max():.0f} API")
print("A–C공, 실제 암상별 감마선 중앙값:", abc.groupby("암상")["감마선_API"].median().round(1).to_dict())
print("D공, k-NN 예측별 감마선 중앙값:  ", d_fix.groupby(pred_D)["감마선_API"].median().round(1).to_dict())


### 해석 (Part B)
- **Q4.** 고른 k와 이유(엘보 · 실루엣 · 지질학적 해석 중 무엇을 따랐나), 각 군집의 이름과 근거(중앙값):
- **Q5 ☆.** k-NN과 k-means가 엇갈린 시료는 몇 개, 어느 깊이인가? 4-5의 결과로 볼 때 어느 쪽이 맞을 것 같은가, 왜?


## 5. Part C ★ — 내 포트폴리오 자료에 모델 하나 (남은 시간 + 집에서)

AI 포트폴리오 작성에 과정에서 활용할 수 있는 **모델 2개 이상 비교 + 정량 성능지표 + 선택 근거**입니다. 오늘은 그중 **첫 모델 하나와 기준선**까지 해 둡니다.

| 내 질문의 답이 … | 모델 | 배운 곳 | 검증 | 기준선(이것보다 나아야 의미가 있다) |
|---|---|---|---|---|
| **수치** (CO₂ 농도, 깊이, 규모, 수위 …) | 회귀 — 직선 · 다항 · 지수 | 9주차 | LOOCV · **시간 자료는 앞뒤로 떼어** | 늘 평균으로 답할 때의 RMSE |
| **범주** (암상, 조구조 환경, 쓰나미 원인, VEI …) | k-NN 분류 | 오늘 Part A | 교차검증 · **같은 화산 · 관측소 · 시추공은 한쪽에** | 늘 가장 많은 범주로 답할 때의 정확도 |
| **라벨 없음** — 무리를 찾고 싶다 (진원 분포, 지화학 시료 …) | k-means 군집 | 오늘 Part B | 실루엣 + **지질학적 해석** | 군집이 이미 알려진 구분(지역 · 암석 · 시기)과 맞는가 |

**자료별 한 줄 주의**
- **지진 목록(기상청 · USGS)**: 진원으로 군집하려면 위경도를 km로 바꾸고(투영) 깊이와 단위를 맞춘 뒤 표준화. 단층 · 섭입대처럼 **길쭉한 무리는 k-means가 잘라 버린다**(개념 페이지 그림 3). 깊이 10 km 고정값은 빼고.
- **마우나로아 CO₂ · 기온**: 수치 목표 → 회귀(직선 vs 2차 vs 계절 항) 비교. 검증은 **마지막 몇 년을 떼어** 맞혀 보기(외삽).
- **GEOROC 지화학**: 조구조 환경 분류(k-NN) — 원소 농도는 **로그**, 같은 화산의 시료는 한쪽에(그룹 단위 검증).
- **SEG 검층**: 오늘과 같은 문제 — **시추공 단위 검증**이 필수.
- **GVP · 쓰나미**: 범주(VEI, 원인) 분류나 회귀 — 먼저 **기록 완전성**(옛 기록의 누락)을 확인.

In [ ]:
# 1) 내 자료 읽기 — 왼쪽 폴더에 올린 뒤 이름을 바꾸세요
# my = pd.read_csv("내자료.csv")
# 2) 질문 한 줄:
# 3) 목표(답)와 특성(입력) 열
# target = "..."; features = ["...", "..."]
# 4) 기준선
# baseline(my[target])
# 5-a) 범주 목표: 같은 화산 · 관측소 · 시추공을 나타내는 열이 있으면 groups="열 이름"
# pred = quick_knn(my, target, features, k=5, groups=None)
# 5-b) 라벨 없음:
# kmeans_scan(my[features]); lab = kmeans_centers(my, k=3, features=features)
# 5-c) 수치 목표: 9주차 노트북의 회귀 · LOOCV 도구를 가져와 씁니다


### 모델 카드 (중간 2에 그대로 옮겨 씀)

| 칸 | 내 답 |
|---|---|
| 질문 한 줄 | |
| 목표(답) · 특성(입력) | |
| 전처리 (단위 · 결측 · 변환) | |
| 기준선 (숫자) | |
| 모델 1 · 검증 방법 · 지표 (숫자) | |
| 결과 한 줄 (기준선과 비교해 숫자로) | |
| 11/30에 비교할 모델 2와 이유 | |

## 6. AI 활용 기록지 (중간 2에 옮겨 씀)

| 도구 | 프롬프트(요지) | 산출물 | 검증(무엇으로 확인했나) | 수정(무엇을 고쳤나) |
|---|---|---|---|---|
| | | | | |
| | | | | |
